# Temporal Fusion Transformer for Marine 2-Day Forecasting
## Training: May 27 - June 24 | Test: June 25-26

### Why TFT for Seasonal Data?
- **Explicit variable selection**: Learns which parameters matter most
- **Temporal attention**: Captures seasonal patterns across time
- **Distribution shift handling**: Better generalization across seasons
- **Interpretability**: Shows feature importance and temporal dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
device = torch.device('cpu')
torch.set_num_threads(8)

print(f"PyTorch device: {device}")
print(f"GPU available: {torch.cuda.is_available()}")

## 1. Load & Prepare Data

In [ ]:
# Load 120-day marine dataset
df_1min = pd.read_csv("marine_data_120days_1min.csv", index_col=0, parse_dates=True)
df_1min = df_1min.drop(columns=['precip_type'], errors='ignore')
df_10min = df_1min.resample("10min").mean().dropna()

print(f"✓ Loaded: {df_10min.shape[0]} rows ({df_10min.shape[0]/(24*6):.1f} days)")
print(f"  Date range: {df_10min.index[0]} to {df_10min.index[-1]}")
print(f"  Shape: {df_10min.shape}")

In [ ]:
# Column mapping: CSV snake_case -> camelCase
CSV_COL_MAP = {
    "air_temp_c": "airTemperature", "air_pressure_hpa": "airPressure",
    "relative_humidity_pct": "relativeHumidity", "dew_point_c": "dewPointTemperature",
    "wind_chill_c": "windChillTemperature", "wind_speed_ms": "windSpeed",
    "wind_direction_deg": "windDirection", "compass_deg": "compass",
    "global_radiation_wm2": "globalRadiation", "current_speed_ms": "currentSpeed",
    "current_direction_deg": "currentDirection", "water_pressure_dbar": "waterPressure",
    "tide_pressure_dbar": "tidePressure", "tidal_level_m": "tideLevel",
    "water_temp_c": "waterTemperature", "conductivity_mscm": "conductivity",
    "salinity_psu": "salinity", "water_temp_quality_c": "waterTemperature_WQ",
    "significant_wave_height_m": "significantWaveHeight", "max_wave_height_m": "maxWaveHeight",
    "water_level_m": "waterLevel", "significant_wave_period_s": "significantWavePeriod",
    "peak_wave_period_s": "peakWaveEnergyPeriod", "zero_crossing_period_s": "zeroCrossingPeriod",
}

df_10min = df_10min.rename(columns=CSV_COL_MAP)
print(f"✓ Columns mapped to camelCase")

In [ ]:
# 18 good parameters + 6 reconstructed duplicates
GOOD_PARAMS = [
    "airTemperature", "airPressure", "relativeHumidity", "dewPointTemperature",
    "windSpeed", "windDirection", "globalRadiation", "currentSpeed", "currentDirection",
    "tideLevel", "waterTemperature", "conductivity", "salinity", "significantWaveHeight",
    "significantWavePeriod", "peakWaveEnergyPeriod", "zeroCrossingPeriod", "compass",
]

DUPLICATES = {
    "windChillTemperature": "airTemperature",
    "tidePressure": "tideLevel",
    "waterPressure": "tideLevel",
    "waterLevel": "tideLevel",
    "waterTemperature_WQ": "waterTemperature",
    "maxWaveHeight": "significantWaveHeight",
}

print(f"✓ Using {len(GOOD_PARAMS)} good parameters (will reconstruct {len(DUPLICATES)} duplicates)")
print(f"  Good: {GOOD_PARAMS[:5]}...")
print(f"  Duplicates: {list(DUPLICATES.keys())[:3]}...")

In [ ]:
# Add temporal features (calendar features)
idx = df_10min.index
df_10min["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
df_10min["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
df_10min["day_sin"] = np.sin(2 * np.pi * idx.day / 30)
df_10min["day_cos"] = np.cos(2 * np.pi * idx.day / 30)
df_10min["month"] = idx.month / 12

TEMPORAL_FEATURES = ["hour_sin", "hour_cos", "day_sin", "day_cos", "month"]
print(f"✓ Added {len(TEMPORAL_FEATURES)} temporal features")

In [ ]:
# Split: 2-day horizon (288 steps), 28-day training (4,032 steps)
horizon_steps = 288  # 2 days @ 10-min resolution
train_steps = 4032   # 28 days @ 10-min resolution
lookback_steps = 288 # 2 days context window

test_start = len(df_10min) - horizon_steps
train_end = test_start
train_start = train_end - train_steps

train_df = df_10min.iloc[train_start:train_end].copy()
test_df = df_10min.iloc[test_start:].copy()

print(f"✓ Data split:")
print(f"  Train: {train_start} to {train_end} ({train_steps} steps = {train_steps//144} days)")
print(f"  Test:  {test_start} to {len(df_10min)} ({horizon_steps} steps = {horizon_steps//144} days)")
print(f"  Train dates: {train_df.index[0]} to {train_df.index[-1]}")
print(f"  Test dates:  {test_df.index[0]} to {test_df.index[-1]}")

In [ ]:
# Standardize (fit on training data only)
scaler = StandardScaler()
train_df[GOOD_PARAMS] = scaler.fit_transform(train_df[GOOD_PARAMS])

# Apply same scaling to test
test_df[GOOD_PARAMS] = scaler.transform(test_df[GOOD_PARAMS])

# Store original test data for later
test_df_orig = df_10min.iloc[test_start:].copy()

print(f"✓ Standardized training data")
print(f"  Mean: {train_df[GOOD_PARAMS].mean().mean():.4f}")
print(f"  Std: {train_df[GOOD_PARAMS].std().mean():.4f}")

## 2. Temporal Fusion Transformer (TFT) Architecture

In [ ]:
class VariableSelectionNetwork(nn.Module):
    """Variable Selection Network: learns which inputs matter most"""
    def __init__(self, input_size, hidden_size=128):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, input_size)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        # x: (batch, time, features)
        x_attn = self.softmax(self.fc2(torch.relu(self.fc1(x))))
        return x * x_attn


class TemporalFusionTransformer(nn.Module):
    """Temporal Fusion Transformer for time series forecasting"""
    def __init__(self, num_params, num_temporal_features, hidden_size=128, n_heads=8, n_layers=2, horizon=288):
        super().__init__()
        
        self.num_params = num_params
        self.hidden_size = hidden_size
        self.horizon = horizon
        
        # Variable selection for input variables
        self.var_select = VariableSelectionNetwork(num_params, hidden_size)
        
        # Variable selection for temporal features
        self.temporal_select = VariableSelectionNetwork(num_temporal_features, hidden_size)
        
        # Input projection
        self.input_proj = nn.Linear(num_params + num_temporal_features, hidden_size)
        
        # Transformer encoder (processes historical context)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size, nhead=n_heads, dim_feedforward=512,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        # Future horizon embedding
        self.horizon_embed = nn.Embedding(horizon, hidden_size)
        
        # Decoder: generate predictions for each horizon step
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_params)
        )
    
    def forward(self, x_past, t_past, t_future):
        # x_past: (batch, lookback, num_params)
        # t_past: (batch, lookback, num_temporal_features)
        # t_future: (batch, horizon, num_temporal_features)
        
        batch_size = x_past.size(0)
        
        # Variable selection
        x_selected = self.var_select(x_past)  # (batch, lookback, num_params)
        t_selected = self.temporal_select(t_past)  # (batch, lookback, num_temporal_features)
        
        # Combine historical inputs with temporal features
        x_combined = torch.cat([x_selected, t_selected], dim=-1)
        x_embedded = self.input_proj(x_combined)
        
        # Transformer encoder: process historical context
        context = self.transformer(x_embedded)  # (batch, lookback, hidden_size)
        context_aggregate = context.mean(dim=1)  # (batch, hidden_size) - aggregate temporal info
        
        # Predict for each horizon step
        predictions = []
        for h in range(self.horizon):
            horizon_feat = self.horizon_embed(torch.tensor(h, device=x_past.device)).unsqueeze(0).expand(batch_size, -1)
            decoder_input = torch.cat([context_aggregate, horizon_feat], dim=-1)
            pred = self.decoder(decoder_input)
            predictions.append(pred)
        
        return torch.stack(predictions, dim=1)  # (batch, horizon, num_params)

print("✓ TFT architecture defined")

## 3. Build Training Data

In [ ]:
class MarineDataset(Dataset):
    """Time series dataset for marine forecasting"""
    def __init__(self, df, params, temporal_features, lookback=288, horizon=288):
        self.df = df
        self.params = params
        self.temporal_features = temporal_features
        self.lookback = lookback
        self.horizon = horizon
        
        self.samples = []
        for i in range(lookback, len(df) - horizon, 2):  # Stride=2 for efficiency
            self.samples.append(i)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        i = self.samples[idx]
        
        # Historical context
        x_past = self.df[self.params].iloc[i - self.lookback:i].values.astype(np.float32)
        t_past = self.df[self.temporal_features].iloc[i - self.lookback:i].values.astype(np.float32)
        
        # Future target
        y_future = self.df[self.params].iloc[i:i + self.horizon].values.astype(np.float32)
        t_future = self.df[self.temporal_features].iloc[i:i + self.horizon].values.astype(np.float32)
        
        return torch.from_numpy(x_past), torch.from_numpy(t_past), torch.from_numpy(t_future), torch.from_numpy(y_future)


# Build training dataset
train_dataset = MarineDataset(
    train_df, GOOD_PARAMS, TEMPORAL_FEATURES,
    lookback=lookback_steps, horizon=horizon_steps
)

print(f"✓ Built training dataset: {len(train_dataset)} samples")
print(f"  Each sample: {lookback_steps} steps lookback → {horizon_steps} steps forecast")

In [ ]:
# Split train/val
n_val = max(1, int(0.1 * len(train_dataset)))
perm = np.random.permutation(len(train_dataset))
val_idx, tr_idx = perm[:n_val], perm[n_val:]

train_indices = [train_dataset.samples[i] for i in tr_idx]
val_indices = [train_dataset.samples[i] for i in val_idx]

train_df_split = train_df.iloc[min(train_indices):max(train_indices)+horizon_steps].copy()
val_df_split = train_df.iloc[min(val_indices):max(val_indices)+horizon_steps].copy()

train_ds_split = MarineDataset(train_df_split, GOOD_PARAMS, TEMPORAL_FEATURES, lookback_steps, horizon_steps)
val_ds_split = MarineDataset(val_df_split, GOOD_PARAMS, TEMPORAL_FEATURES, lookback_steps, horizon_steps)

train_loader = DataLoader(train_ds_split, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds_split, batch_size=32, shuffle=False)

print(f"✓ Train/Val split:")
print(f"  Train: {len(train_ds_split)} batches")
print(f"  Val: {len(val_ds_split)} batches")

## 4. Train TFT

In [ ]:
# Initialize model
model = TemporalFusionTransformer(
    num_params=len(GOOD_PARAMS),
    num_temporal_features=len(TEMPORAL_FEATURES),
    hidden_size=128,
    n_heads=8,
    n_layers=2,
    horizon=horizon_steps
).to(device)

print(f"✓ TFT Model initialized")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()
patience = 15

train_losses, val_losses = [], []
best_val_loss = float('inf')
best_state = None
wait = 0

print("Starting training...\n")
t_start = time.time()

for epoch in range(50):
    # Train
    model.train()
    epoch_loss = 0
    for x_past, t_past, t_future, y_future in train_loader:
        x_past, t_past, t_future, y_future = x_past.to(device), t_past.to(device), t_future.to(device), y_future.to(device)
        
        optimizer.zero_grad()
        y_pred = model(x_past, t_past, t_future)
        loss = criterion(y_pred, y_future)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)
    
    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x_past, t_past, t_future, y_future in val_loader:
            x_past, t_past, t_future, y_future = x_past.to(device), t_past.to(device), t_future.to(device), y_future.to(device)
            y_pred = model(x_past, t_past, t_future)
            val_loss += criterion(y_pred, y_future).item()
    
    val_loss = val_loss / len(val_loader)
    val_losses.append(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss - 1e-6:
        best_val_loss = val_loss
        wait = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
    
    if (epoch + 1) % 10 == 0 or wait >= patience:
        print(f"Epoch {epoch+1:2d}/50 | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Wait: {wait}/15")
    
    if wait >= patience:
        print(f"→ Early stop at epoch {epoch+1}")
        break

if best_state:
    model.load_state_dict(best_state)

t_train = time.time() - t_start
print(f"\n✓ Training complete: {t_train:.1f}s")

In [ ]:
# Plot training history
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, label='Train Loss', marker='o', markersize=3, alpha=0.7)
ax.plot(val_losses, label='Val Loss', marker='s', markersize=3, alpha=0.7)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('TFT Training Progress', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best validation loss: {best_val_loss:.6f}")

## 5. Evaluate on Test Window (June 25-26)

In [ ]:
# Prepare test data
model.eval()

# Last lookback window from training data
x_test = train_df[GOOD_PARAMS].iloc[-lookback_steps:].values.astype(np.float32)
t_test = train_df[TEMPORAL_FEATURES].iloc[-lookback_steps:].values.astype(np.float32)

# Future temporal features
t_future_test = test_df[TEMPORAL_FEATURES].iloc[:horizon_steps].values.astype(np.float32)

# Convert to tensors
x_test_t = torch.from_numpy(x_test).unsqueeze(0).to(device)
t_test_t = torch.from_numpy(t_test).unsqueeze(0).to(device)
t_future_t = torch.from_numpy(t_future_test).unsqueeze(0).to(device)

# Forecast
t_infer = time.time()
with torch.no_grad():
    y_pred_norm = model(x_test_t, t_test_t, t_future_t)[0].cpu().numpy()
t_infer = time.time() - t_infer

print(f"✓ Forecast generated in {t_infer*1000:.2f}ms")
print(f"  Shape: {y_pred_norm.shape}")

In [ ]:
# Inverse standardization
y_pred = scaler.inverse_transform(y_pred_norm)

# Reconstruct duplicates from twins using linear regression
from sklearn.linear_model import LinearRegression

# Fit reconstruction models on training data
recon_models = {}
for dup_col, twin_col in DUPLICATES.items():
    twin_idx = GOOD_PARAMS.index(twin_col)
    X_train_dup = train_df_orig[twin_col].values.reshape(-1, 1)
    y_train_dup = train_df_orig[dup_col].values
    
    model_dup = LinearRegression()
    model_dup.fit(X_train_dup, y_train_dup)
    recon_models[dup_col] = model_dup

# Reconstruct for test window
y_pred_all = np.hstack([y_pred, np.zeros((horizon_steps, len(DUPLICATES)))])
for k, (dup_col, twin_col) in enumerate(DUPLICATES.items()):
    twin_idx = GOOD_PARAMS.index(twin_col)
    y_pred_all[:, len(GOOD_PARAMS) + k] = recon_models[dup_col].predict(y_pred[:, twin_idx:twin_idx+1]).flatten()

print(f"✓ Reconstructed {len(DUPLICATES)} duplicate parameters")

In [ ]:
# Compute metrics
y_true = test_df_orig[GOOD_PARAMS].iloc[:horizon_steps].values
y_true_all = test_df_orig[GOOD_PARAMS + list(DUPLICATES.keys())].iloc[:horizon_steps].values

# Persistence baseline
last_obs = df_10min[GOOD_PARAMS].iloc[-horizon_steps - 1].values
y_persist = np.tile(last_obs, (horizon_steps, 1))
y_persist_all = np.tile(df_10min[GOOD_PARAMS + list(DUPLICATES.keys())].iloc[-horizon_steps - 1].values, (horizon_steps, 1))

# Compute skill
mae_good = mean_absolute_error(y_true, y_pred)
mae_good_persist = mean_absolute_error(y_true, y_persist)
skill_good = (1 - mae_good / mae_good_persist) * 100 if mae_good_persist > 0 else 0

mae_all = mean_absolute_error(y_true_all, y_pred_all)
mae_all_persist = mean_absolute_error(y_true_all, y_persist_all)
skill_all = (1 - mae_all / mae_all_persist) * 100 if mae_all_persist > 0 else 0

print(f"\n" + "="*70)
print(f"PERFORMANCE METRICS (2-Day Forecast: June 25-26)")
print(f"="*70)
print(f"\nGood 18 Parameters:")
print(f"  MAE: {mae_good:.4f}")
print(f"  Persistence MAE: {mae_good_persist:.4f}")
print(f"  Skill: {skill_good:+.1f}%")
print(f"\nAll 24 Parameters (incl. reconstructed):")
print(f"  MAE: {mae_all:.4f}")
print(f"  Persistence MAE: {mae_all_persist:.4f}")
print(f"  Skill: {skill_all:+.1f}%")
print(f"="*70)

In [ ]:
# Per-parameter metrics
metrics_list = []
all_params_list = GOOD_PARAMS + list(DUPLICATES.keys())

for j, p in enumerate(all_params_list):
    y_t = y_true_all[:, j]
    y_p = y_pred_all[:, j]
    y_pers = y_persist_all[:, j]
    
    mae = mean_absolute_error(y_t, y_p)
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    mae_pers = mean_absolute_error(y_t, y_pers)
    skill = (1 - mae / mae_pers) * 100 if mae_pers > 0 else 0
    
    is_dup = "*" if p in DUPLICATES else " "
    metrics_list.append({
        "Parameter": p + is_dup,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "Skill_%": round(skill, 1),
        "Type": "Reconstructed" if is_dup == "*" else "Modeled"
    })

metrics_df = pd.DataFrame(metrics_list)
print(f"\nPer-Parameter Performance (* = reconstructed):")
print(metrics_df.to_string(index=False))
print(f"\nNote: * indicates reconstructed duplicates from linear regression")

In [ ]:
# Top and bottom performers
print(f"\n{'='*70}")
print(f"TOP 5 PARAMETERS:")
print(f"{'='*70}")
for _, row in metrics_df.nlargest(5, "Skill_%").iterrows():
    print(f"  {row['Parameter']:30s} {row['Skill_%']:+7.1f}%  MAE: {row['MAE']:.4f}")

print(f"\n{'='*70}")
print(f"BOTTOM 5 PARAMETERS:")
print(f"{'='*70}")
for _, row in metrics_df.nsmallest(5, "Skill_%").iterrows():
    print(f"  {row['Parameter']:30s} {row['Skill_%']:+7.1f}%  MAE: {row['MAE']:.4f}")

## 6. Visualize Predictions

In [ ]:
# Select parameters to plot
plot_params = [
    "airTemperature", "waterTemperature", "windSpeed",
    "salinity", "tideLevel", "significantWaveHeight"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, param in enumerate(plot_params):
    j = all_params_list.index(param)
    
    ax = axes[idx]
    time_idx = np.arange(horizon_steps) / 144  # Convert to days
    
    ax.plot(time_idx, y_true_all[:, j], 'o-', label='Actual', linewidth=2, markersize=4, alpha=0.7)
    ax.plot(time_idx, y_pred_all[:, j], 's--', label='Forecast', linewidth=2, markersize=4, alpha=0.7)
    ax.plot(time_idx, y_persist_all[:, j], '^:', label='Persistence', linewidth=1.5, markersize=3, alpha=0.5)
    
    skill_val = metrics_df[metrics_df['Parameter'].str.strip() == param]['Skill_%'].values[0]
    ax.set_title(f'{param}\nSkill: {skill_val:+.1f}%', fontsize=12, fontweight='bold')
    ax.set_xlabel('Days', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.suptitle('TFT 2-Day Forecast: June 25-26', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Skill distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Modeled vs Reconstructed
modeled = metrics_df[metrics_df['Type'] == 'Modeled']['Skill_%'].values
recon = metrics_df[metrics_df['Type'] == 'Reconstructed']['Skill_%'].values

ax1.bar(['Modeled (18)', 'Reconstructed (6)'], 
        [modeled.mean(), recon.mean()],
        color=['#1f77b4', '#ff7f0e'],
        alpha=0.7,
        edgecolor='black',
        linewidth=2)
ax1.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Baseline')
ax1.set_ylabel('Mean Skill (%)', fontsize=11)
ax1.set_title('Modeled vs Reconstructed', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
ax1.legend()

# All parameters skill distribution
ax2.barh(metrics_df['Parameter'], metrics_df['Skill_%'], 
         color=['#1f77b4' if m == 'Modeled' else '#ff7f0e' for m in metrics_df['Type']],
         alpha=0.7,
         edgecolor='black')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Skill (%)', fontsize=11)
ax2.set_title('Per-Parameter Skill', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 7. Summary & Results

In [ ]:
# Save results
results_summary = pd.DataFrame([
    {
        "Model": "Temporal Fusion Transformer",
        "Skill_Good_18": skill_good,
        "Skill_All_24": skill_all,
        "MAE_Good_18": mae_good,
        "MAE_All_24": mae_all,
        "Training_Time_s": t_train,
        "Inference_Time_ms": t_infer * 1000,
        "Training_Dates": f"{train_df.index[0]} to {train_df.index[-1]}",
        "Test_Dates": f"{test_df.index[0]} to {test_df.index[-1]}",
    }
])

results_summary.to_csv("tft_2day_results.csv", index=False)
metrics_df.to_csv("tft_2day_metrics.csv", index=False)

print(f"\n✓ Results saved:")
print(f"  - tft_2day_results.csv")
print(f"  - tft_2day_metrics.csv")

print(f"\n{'='*70}")
print(f"FINAL SUMMARY")
print(f"{'='*70}")
print(results_summary.to_string(index=False))
print(f"{'='*70}")

In [ ]:
print(f"\n📊 KEY FINDINGS:")
print(f"\n1. MODELING APPROACH:")
print(f"   • Temporal Fusion Transformer with variable selection network")
print(f"   • Trained on 18 good parameters only")
print(f"   • Duplicates reconstructed via linear regression on twins")
print(f"\n2. DATA WINDOW:")
print(f"   • Training: May 27 - June 24 (28 days)")
print(f"   • Testing: June 25-26 (2 days)")
print(f"   • Seasonal context: Late spring → early summer transition")
print(f"\n3. PERFORMANCE:")
print(f"   • Good-18 Skill: {skill_good:+.1f}%")
print(f"   • All-24 Skill: {skill_all:+.1f}%")
print(f"   • Training Time: {t_train:.1f}s")
print(f"   • Inference Time: {t_infer*1000:.2f}ms")
print(f"\n4. INSIGHTS:")
print(f"   • TFT handles temporal dependencies via multi-head attention")
print(f"   • Variable selection learns feature importance")
print(f"   • Reconstruction adds noise (All-24 < Good-18)")
print(f"\n5. NEXT STEPS:")
print(f"   • Train on 6 horizons (2-7 days) with 14× ratio rule")
print(f"   • Compare vs other models (SARIMA, Prophet)")
print(f"   • Fine-tune hyperparameters for seasonal data")